<a href="https://colab.research.google.com/github/n2pakbong/boundary-adapted-pinns/blob/main/colab/periodic_committor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Periodic two-well committor experiment

## Why this problem?

A committor is the probability that a stochastic trajectory reaches a target set before it returns to a source set. It is a central quantity in transition path theory and metastability: it describes which way a transition is likely to go from each starting point. This example tests a boundary-adapted neural representation on a small periodic problem with two separate interior boundaries.

This is a **committor** experiment, not a mean escape time experiment. The committor answers a probability question; a mean escape time answers how long a process typically takes to hit a set. The mathematical motivation for boundary-adapted PINNs is discussed in the accompanying paper, [*Boundary-Adapted PINNs for Elliptic Dirichlet Problems*](https://arxiv.org/abs/2607.19167).

## Stochastic dynamics and the committor equation

Let $X_t$ follow overdamped Langevin dynamics on the flat torus $\mathbb{T}^2=[0,1)^2$:
$$
dX_t=-\nabla V(X_t)\,dt+\sqrt{2\varepsilon}\,dW_t\pmod 1,
\qquad
V(x_1,x_2)=a[1-\cos(4\pi(x_1-1/4))]+b[1-\cos(2\pi x_2)].
$$
The periodic disks $A$ and $B$ have centers $c_A=(1/4,0)$ and $c_B=(3/4,0)$ and common radius $r$. For their first hitting times $\tau_A$ and $\tau_B$, the forward committor is
$$
q(x)=\mathbb{P}_x(\tau_B<\tau_A).
$$
It is the probability of reaching $B$ before $A$ when starting at $x$. The diffusion generator is
$$
\mathcal{L}f=-\nabla V\cdot\nabla f+\varepsilon\Delta f.
$$
The Markov property implies that $q$ is harmonic with respect to this generator:
$$
\mathcal{L}q=0\quad\text{in }D=\mathbb{T}^2\setminus(A\cup B),
\qquad q|_{\partial A}=0,\quad q|_{\partial B}=1.
$$
The default parameters are $a=1$, $b=0.6$, and $\varepsilon=0.4$.

## Exact boundary values from a smooth geometry factor

For a periodic disk centered at $c_j$, the code uses the smooth periodic defining function
$$
\omega_j(x)=4\sum_{\ell=1}^2\sin^2(\pi(x_\ell-c_{j\ell}))-r^2.
$$
It is negative inside the disk, zero on its boundary, and positive in the exterior domain. The normalized defining function is
$$
d_j(x)=\frac{\omega_j(x)}{\sqrt{\omega_j(x)^2+|\nabla\omega_j(x)|^2}}.
$$
On the boundary, $d_j=0$ and $|\nabla d_j|=1$. In $D$, both $d_A$ and $d_B$ are positive. Define the boundary lift $g$ and the two-boundary factor $\rho$ by
$$
g(x)=\frac{d_A(x)}{d_A(x)+d_B(x)},\qquad
\rho(x)=\frac{d_A(x)d_B(x)}{\sqrt{d_A(x)^2+d_B(x)^2}}.
$$
The network represents
$$
q_\theta(x)=g(x)+\rho(x)N_\theta(x).
$$
On $\partial A$, $g=0$ and $\rho=0$; on $\partial B$, $g=1$ and $\rho=0$. Therefore the two Dirichlet values hold for every choice of network weights. Near either boundary, $\rho$ has the first-order scaling of that boundary's normalized defining function. The simpler product $d_Ad_B$ also vanishes on both boundaries, but its normal scaling depends on the distance to the other boundary.

All three networks receive sine and cosine coordinates $(\sin(2\pi x_1),\sin(2\pi x_2),\cos(2\pi x_1),\cos(2\pi x_2))$, so their outputs are periodic by construction.

## Five methods and their losses

All methods solve the committor boundary-value problem above and use the same collocation points and Monte Carlo labels within one comparison. The shared smooth lift and first-order boundary factor are
$$
g=\frac{d_A}{d_A+d_B},\qquad \rho=\frac{d_Ad_B}{\sqrt{d_A^2+d_B^2}}.
$$

The exact-boundary trial functions are
$$
q_\theta=g+\rho^\alpha N_\theta.
$$
The boundary-adapted method uses $\alpha=1$; over-enforced and under-enforced use $\alpha=3/2$ and $1/2$. All three satisfy $q=0$ on $\partial A$ and $q=1$ on $\partial B$ exactly. Their optimized losses have no boundary-residual term.

The standard PINN is the plain periodic network $q_\theta=N_\theta$. It uses the ordinary boundary residual
$$
L_{\partial}=\operatorname{mean}_{x\in\partial A}q_\theta(x)^2+
\operatorname{mean}_{x\in\partial B}(q_\theta(x)-1)^2.
$$
This is the only method with that standard boundary penalty.

Every method also uses pointwise Monte Carlo committor estimates as a solution-value regularizer. Starting at an anchor $z_i$, independent Euler–Maruyama paths follow
$$
X_{n+1}=\bigl[X_n-\nabla V(X_n)\Delta t+
\sqrt{2\varepsilon\Delta t}\,\xi_n\bigr]\pmod 1,\qquad \xi_n\sim\mathcal N(0,I_2),
$$
until each path enters $A$ or $B$. The estimate $\widehat q_i$ is the fraction of resolved paths that hit $B$ first. Unresolved paths are counted and are never treated as hits of $A$; if the configured minimum resolution is not met, training stops with an error. The common data loss is
$$
L_{\rm MC}=\frac1{N_M}\sum_{i=1}^{N_M}
\left(q_\theta(z_i)-\widehat q_i\right)^2.
$$
The saved Monte Carlo settings include the number of anchors and paths, time step, maximum path length, seed, resolved fractions, and conditional binomial standard errors from resolved paths. Any residual censoring below the configured threshold can still bias the labels; see the experiment guide. This is the committor version of the pointwise Monte Carlo solution regularizer used in the mean-escape-time notebooks; it does not turn this example into a mean escape time calculation.

The variational/Nitsche method uses a plain periodic network and the reversible density $m=e^{-V/\varepsilon}$, for which
$$
\mathcal Lq=0\quad\Longleftrightarrow\quad
\nabla\cdot(\varepsilon m\nabla q)=0.
$$
With periodic Gaussian test functions $\phi_j$, and $n_D$ the outward normal from $D$ into the removed sets, its weak residual is
$$
R_j=\varepsilon\int_Dm\nabla q_\theta\cdot\nabla\phi_j\,dx
-\varepsilon\int_{\partial D}m\,\partial_{n_D}q_\theta\,\phi_j\,ds
-\varepsilon\int_{\partial D}m(q_\theta-g)\,\partial_{n_D}\phi_j\,ds.
$$
The last term is the symmetric Nitsche term; the separate stabilization moment is
$$
P_j=\frac{\gamma\varepsilon}{h}
\int_{\partial D}m(q_\theta-g)\phi_j\,ds.
$$
It minimizes the mean squares of the weak residuals $R_j$ and stabilization moments $P_j$, plus the common Monte Carlo data loss. This Nitsche stabilization belongs to this variational method. It is not added to any exact-boundary method.

For each method, every active raw loss component $L_c$ has a trainable scale $s_c$. The optimized objective is
$$
J(\theta,s)=\sum_c\left(e^{-2s_c}L_c(\theta)+s_c\right).
$$
For example, the standard PINN uses PDE, Monte Carlo, and boundary-residual losses; each exact-boundary PINN uses PDE and Monte Carlo losses; the variational method uses weak-residual, Nitsche-stabilization, and Monte Carlo losses. The notebook reports raw losses, weighted losses, and current effective weights $e^{-2s_c}$ during training and at completion. Because the trainable scale offsets are part of the objective, that optimization objective can be negative; the reported raw components are the nonnegative residual/data quantities.

The full derivation, quadrature details, method-to-loss table, and references are in the experiment guide at ../experiments/committor/periodic_two_well/README.md.

## Further reading

- M. Raissi, P. Perdikaris, and G. E. Karniadakis, “Physics-informed neural networks,” *Journal of Computational Physics* 378 (2019), 686–707. [doi:10.1016/j.jcp.2018.10.045](https://doi.org/10.1016/j.jcp.2018.10.045).
- N. Sukumar and A. Srivastava, “Exact imposition of boundary conditions with distance functions in physics-informed deep neural networks,” *Computer Methods in Applied Mechanics and Engineering* 389 (2022), 114333. [doi:10.1016/j.cma.2021.114333](https://doi.org/10.1016/j.cma.2021.114333).
- S. Berrone, C. Canuto, M. Pintore, and N. Sukumar, “Enforcing Dirichlet boundary conditions in physics-informed neural networks and variational physics-informed neural networks,” *Heliyon* 9(8) (2023), e18820. [doi:10.1016/j.heliyon.2023.e18820](https://doi.org/10.1016/j.heliyon.2023.e18820).
- N. Tepakbong, J. Fan, X. Zhou, and D.-X. Zhou, “Boundary-Adapted PINNs for Elliptic Dirichlet Problems,” [arXiv:2607.19167](https://arxiv.org/abs/2607.19167) (2026).
- Y. Khoo, J. Lu, and L. Ying, “Solving for high dimensional committor functions using artificial neural networks,” [arXiv:1802.10275](https://arxiv.org/abs/1802.10275) (2018).
- Z. Song, M. K. Cameron, and H. Yang, “A Finite Expression Method for Solving High-Dimensional Committor Problems,” [arXiv:2306.12268](https://arxiv.org/abs/2306.12268) (2023).
## What each stage does

| Stage | Purpose | Compute | Saved output |
|---|---|---|---|
| Setup and CPU tests | Install the selected code revision and run mathematical and tiny training tests. | CPU | None. Prints the installed code revision. |
| CPU preflight | Exercise all five methods, Monte Carlo wiring, reference solve, summaries, and plots with tiny test settings. | CPU | Temporary Colab storage only. |
| GPU comparison | Estimate common Monte Carlo labels once on CPU, then train five methods for 6,000 epochs each and compute 128/256-grid references. | GPU for neural-network training; CPU for paths, finite differences, and plots. | One timestamped run folder in Drive. |
| Inspect | Display training components and curves, Sobolev errors, and solution previews. | CPU/display | No retraining. |
| Saved-model evaluation | Recompute absolute and relative L2, H1, and H2 errors from saved checkpoints. | CPU only | Adds a JSON report beside the existing run. |
| Resume | Continue an interrupted run with its saved configuration, labels, checkpoints, and random state. | GPU for remaining training | Same run folder. |
## Where results go

The GPU cell prints the complete Drive path before training starts. That folder is the durable record. The notebook guide explains every output and keeps the math and estimator settings beside the code.

- comparison_config.json contains the exact method list, model, problem, optimizer, Monte Carlo, and Nitsche settings.
- comparison_metadata.json identifies the run time and installed code revision.
- monte_carlo_labels.npz and monte_carlo_labels.json hold the common anchors, estimates, uncertainty diagnostics, and estimator settings used for all five methods.
- summary.json contains final raw training-loss components, best epoch/objective, boundary diagnostics, and absolute/relative L2, H1, and H2 errors against the finite-difference reference.
- Each method folder has metrics.jsonl for live loss history, final_training_metrics.json for final raw/weighted loss values and adaptive weights, best.pt for evaluation, and last.pt for resuming.
- losses.png plots the total objective and each raw loss component. committor_*.png are diagnostic previews, not paper-ready figures.

The completed older three-method pilot is not rewritten. Its CPU evaluation can report its saved Sobolev errors and recorded PDE/boundary losses, but it cannot retroactively acquire the Monte Carlo term or missing methods. A report for that run marks the data term as absent. The five-method configuration is a new exploratory run and is not a request to rerun training now.

## 1. Setup and CPU tests — CPU only

Run this cell first. When prompted for a branch, press Enter for the normal `main` workflow. To test this open pull request before merging, enter `committor-notebook-review`. Setup fetches that branch, installs the package, and runs the mathematical and tiny training tests on CPU. The printed code revision identifies exactly what was installed. No GPU or Drive storage is used.

In [ ]:
# CPU setup: select a Git branch, install it, and run mathematical and tiny training tests.
from pathlib import Path
import base64
import os
import subprocess

from google.colab import userdata

CPU_PREFLIGHT_PASSED = False

def run_logged(command, log_path, append=False):
    """Show output and offer the complete log as a download if the command fails."""
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open("a" if append else "w", encoding="utf-8") as log:
        process = subprocess.Popen(
            command,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            env={**os.environ, "PYTHONUNBUFFERED": "1"},
        )
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()
        return_code = process.wait()
    if return_code:
        print(f"\nCommand failed (exit {return_code}). The full traceback is above.")
        try:
            from google.colab import files
            files.download(str(log_path))
            print("Attach the downloaded console.log if you need help diagnosing the error.")
        except Exception as download_error:
            print(f"Log download did not start: {download_error}. Copy the traceback above.")
        raise RuntimeError("Command failed. The traceback and downloadable log are above.")

REPO = Path('/content/boundary-adapted-pinns')
REPO_URL = 'https://github.com/n2pakbong/boundary-adapted-pinns.git'
BRANCH = input('Git branch to install [main]: ').strip() or 'main'

if REPO.exists() and not (REPO / '.git').is_dir():
    raise RuntimeError(f'{REPO} exists but is not a Git clone. Rename that temporary folder, then rerun setup.')

token = None
try:
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    # Publicly readable repositories can be cloned without authentication.
    pass

git_auth = ['git']
if token:
    auth = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
    git_auth += ['-c', f'http.extraheader=AUTHORIZATION: basic {auth}']

try:
    if not REPO.exists():
        subprocess.run([
            *git_auth, 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(REPO)
        ], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO), 'remote', 'set-url', 'origin', REPO_URL], check=True)
        subprocess.run([
            *git_auth, '-C', str(REPO), 'fetch', '--depth', '1', 'origin', BRANCH
        ], check=True)
        subprocess.run(['git', '-C', str(REPO), 'checkout', '-B', BRANCH, 'FETCH_HEAD'], check=True)
except subprocess.CalledProcessError as error:
    raise RuntimeError(
        'Could not fetch the selected branch. Check the branch name and repository access, then rerun Setup.'
    ) from error
finally:
    del token
    if 'auth' in globals():
        del auth

%cd /content/boundary-adapted-pinns
subprocess.run(['python', '-m', 'pip', 'install', '-q', '-e', '.[dev]'], check=True)
subprocess.run(['python', '-m', 'pytest', '-q'], check=True)
revision = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('Setup and CPU tests passed. Code revision:', revision)


## 2. CPU preflight — full pipeline check

This one-epoch check runs all five methods, including a tiny fixed-anchor Monte Carlo sample and Nitsche weak form, entirely on CPU. It uses tiny networks and coarse reference grids to check code wiring, not numerical accuracy. Outputs go to temporary Colab storage only. Repeating the cell replaces the previous temporary preflight. If a command fails, its traceback appears here and Colab offers the complete console log for download. The GPU comparison cell stays blocked until this succeeds in the current runtime.

In [ ]:
# CPU only: run the full pipeline in temporary Colab storage.
from pathlib import Path
import shutil

CPU_PREFLIGHT_PASSED = False
preflight_dir = Path('/content/periodic_committor_preflight')
if preflight_dir.exists():
    shutil.rmtree(preflight_dir)
preflight_dir.mkdir()
PREFLIGHT_ROOT = str(preflight_dir)
print('CPU preflight outputs (temporary Colab storage):', PREFLIGHT_ROOT)

run_logged([
    'periodic-committor',
    'experiments/committor/periodic_two_well/configs/cpu_preflight.json',
    '--output-dir', PREFLIGHT_ROOT,
], preflight_dir / 'console.log')
if not (preflight_dir / 'summary.json').is_file():
    raise RuntimeError('CPU preflight produced no summary.json; check the command output above.')
CPU_PREFLIGHT_PASSED = True
print('CPU preflight passed. The GPU pilot is now available in this runtime.')

## 3. GPU comparison — first substantive five-method experiment

The GPU cell runs only after the CPU preflight succeeds. It uses 64-wide, 4-hidden-layer networks, 1,024 strong-form collocation points, 6,000 epochs per configured method, and 128/256-grid references. The Monte Carlo committor labels are generated once on CPU and shared across methods; only neural-network training uses GPU. The Nitsche config separately specifies volume points, Gaussian test functions, boundary points, length scale, stabilization parameter, and mesh scale. Drive is mounted when training begins, and the cell saves its log and restartable outputs in one dated folder. This is a single-seed exploratory comparison, not a paper result.

In [ ]:
# GPU required: full five-method paper-formulation committor comparison.
if not globals().get('CPU_PREFLIGHT_PASSED', False):
    raise RuntimeError('GPU pilot blocked: the CPU preflight has not passed in this runtime. Fix its error first.')
from datetime import datetime
from pathlib import Path
import subprocess
import torch

if not torch.cuda.is_available():
    raise RuntimeError('The GPU pilot requires a T4 GPU. Select it under Runtime → Change runtime type, then rerun setup and preflight.')

from google.colab import drive
drive.mount('/content/drive')

run_name = 'periodic_committor_' + datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_ROOT = f'/content/drive/MyDrive/boundary-adapted-pinns-runs/periodic_committor/{run_name}'
Path(RUN_ROOT).mkdir(parents=True, exist_ok=False)
print('GPU pilot results:', RUN_ROOT)

run_logged([
    'periodic-committor',
    'experiments/committor/periodic_two_well/configs/gpu_pilot.json',
    '--output-dir', RUN_ROOT,
], Path(RUN_ROOT) / 'console.log')

## 4. Inspect the finished comparison

Run this after the GPU cell finishes. It performs no training. It displays both the solution-error table and the final raw training-loss components, then shows the saved component curves and diagnostic solution figures.

For an older completed three-method pilot, the saved solution errors can still be evaluated, but its missing Monte Carlo term and methods cannot be reconstructed from checkpoints.

In [ ]:
import json
from pathlib import Path
from IPython.display import Image, display
import pandas as pd

with open(f'{RUN_ROOT}/summary.json') as handle:
    summary = json.load(handle)

method_summaries = summary.get('methods', {})
accuracy_fields = [
    'grid_l2_abs', 'grid_l2_rel',
    'grid_h1_abs', 'grid_h1_rel',
    'grid_h1_seminorm_abs', 'grid_h1_seminorm_rel',
    'grid_h2_abs', 'grid_h2_rel',
    'grid_h2_seminorm_abs', 'grid_h2_seminorm_rel',
]
if method_summaries and all(
    all(field in metrics for field in accuracy_fields)
    for metrics in method_summaries.values()
):
    display(pd.DataFrame(method_summaries).T[accuracy_fields])
else:
    print(
        'This run summary predates the full Sobolev error report. '
        'Run Step 5 to calculate absolute and relative L2, H1, and H2 errors '
        'from its saved checkpoints on CPU; no GPU retraining is needed.'
    )

print('Final raw training losses (not solution-error metrics):')
training_rows = {}
for method, metrics in method_summaries.items():
    training = metrics.get('final_training_metrics', {})
    raw_components = training.get('raw_loss_components', {})
    if not raw_components:
        metrics_path = Path(RUN_ROOT) / method / 'metrics.jsonl'
        if metrics_path.is_file():
            lines = [line for line in metrics_path.read_text().splitlines() if line.strip()]
            if lines:
                last = json.loads(lines[-1])
                raw_components = {
                    key: value for key, value in last.items()
                    if key.endswith('_loss') or key.endswith('_mse')
                }
                training = {
                    'final_training_objective': last.get('objective', last.get('loss')),
                    'raw_loss_components': raw_components,
                }
    training_rows[method] = {
        'final objective': training.get('final_training_objective'),
        **raw_components,
    }
display(pd.DataFrame(training_rows).T)

print('Finite-difference grid comparison:')
print(json.dumps(summary.get('reference_convergence', {}), indent=2))
display(Image(filename=f'{RUN_ROOT}/losses.png'))
for method in method_summaries:
    figure = Path(RUN_ROOT) / f'committor_{method}.png'
    if figure.is_file():
        display(Image(filename=str(figure)))

## 5. Evaluate saved models against the finite-difference solution — CPU only

This step reloads each method's saved best checkpoint, computes absolute and relative L2, H1, and grid-estimated H2 errors against the saved finite-difference reference, and checks boundary RMSE on actual level-set points. It uses CPU only; it does not train, use GPU, or overwrite summary.json.

The report is displayed here and saved as saved_model_evaluation.json in the existing run folder. It also displays the final training-loss components recorded during the run. For the older three-method pilot, the report explicitly marks Monte Carlo data loss as unavailable because that run did not use it. New five-method runs save their training-code revision and estimator settings automatically.

In [ ]:
# CPU only: compare saved best.pt checkpoints with the FDM target; this does not train.
from pathlib import Path
import json
import subprocess
from google.colab import drive
from IPython.display import display
import pandas as pd

if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')

if 'RUN_ROOT' not in globals() or not Path(RUN_ROOT, 'comparison_config.json').is_file():
    RUN_ROOT = input('Paste the completed GPU pilot Drive folder path: ').strip()

if not Path(RUN_ROOT, 'comparison_config.json').is_file():
    raise FileNotFoundError(f'No comparison_config.json found in {RUN_ROOT}')

command = ['periodic-committor', '--output-dir', str(RUN_ROOT), '--evaluate-saved-models']
subprocess.run(command, check=True)

report_path = Path(RUN_ROOT) / 'saved_model_evaluation.json'
report = json.loads(report_path.read_text(encoding='utf-8'))
print('Training code revision:', report.get('training_git_revision') or 'not recorded for this older run')
print('FDM reference:', report['reference_file'])
display_fields = [
    'grid_l2_abs', 'grid_l2_rel',
    'grid_h1_abs', 'grid_h1_rel',
    'grid_h1_seminorm_abs', 'grid_h1_seminorm_rel',
    'grid_h2_abs', 'grid_h2_rel',
    'grid_h2_seminorm_abs', 'grid_h2_seminorm_rel',
    'source_boundary_rmse', 'target_boundary_rmse',
]
display(pd.DataFrame(report['methods']).T[display_fields])

print('Final training-loss components from the saved run:')
loss_rows = {}
for method, result in report['methods'].items():
    training = result.get('final_training_metrics', {})
    loss_rows[method] = {
        'final objective': training.get('final_training_objective'),
        **training.get('raw_loss_components', {}),
        'MC data term used': training.get('monte_carlo_labels') is not None,
    }
display(pd.DataFrame(loss_rows).T)
print('Full saved report:', report_path)

## Resume an interrupted GPU pilot

Use this only if the GPU pilot started and then stopped before finishing. In a fresh runtime, run setup and pass the CPU preflight first. Then paste the Drive folder path printed by the interrupted pilot. The saved checkpoints continue the existing run. The resume cell is locked until the CPU preflight passes.

In [ ]:
# Run this only if the GPU pilot stopped before it finished.
if not globals().get('CPU_PREFLIGHT_PASSED', False):
    raise RuntimeError('Resume blocked: rerun setup and pass the CPU preflight in this runtime first.')
# Paste the exact Drive folder path printed at the start of that pilot.
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

RUN_ROOT = input('Paste the interrupted pilot Drive folder path, then press Enter: ').strip()
if not Path(RUN_ROOT, 'comparison_config.json').is_file():
    raise FileNotFoundError('That folder is missing comparison_config.json. Paste the exact folder path printed by the interrupted pilot.')

run_logged([
    'periodic-committor',
    'experiments/committor/periodic_two_well/configs/gpu_pilot.json',
    '--output-dir', RUN_ROOT,
    '--resume',
], Path(RUN_ROOT) / 'console.log', append=True)
print('The interrupted pilot has resumed and completed.')

## One-time Drive cleanup (optional)

Earlier notebook versions saved CPU checks and some failed GPU attempts to Drive. Run the next cell once, while no experiment is running, to remove the timestamped CPU preflight folders. It also removes a GPU attempt folder only if it contains nothing beyond a console log and run metadata. Any GPU folder containing reference data, checkpoints, figures, or other files is preserved. Current and future CPU preflights stay in temporary Colab storage.

In [ ]:
# Optional: remove old CPU checks and empty failed GPU attempts created by this notebook.
from pathlib import Path
import re
import shutil
from google.colab import drive

drive.mount('/content/drive')
old_run_root = Path('/content/drive/MyDrive/boundary-adapted-pinns-runs/periodic_committor')
known_startup_files = {'console.log', 'comparison_config.json', 'comparison_metadata.json'}
old_cleanup_targets = []
if old_run_root.is_dir():
    for folder in old_run_root.iterdir():
        if not folder.is_dir():
            continue
        if re.fullmatch(r'preflight_\d{8}_\d{6}', folder.name):
            old_cleanup_targets.append(folder)
        elif re.fullmatch(r'periodic_committor_\d{8}_\d{6}', folder.name):
            if all(child.is_file() and child.name in known_startup_files for child in folder.iterdir()):
                old_cleanup_targets.append(folder)
for folder in sorted(old_cleanup_targets):
    shutil.rmtree(folder)
    print('Removed:', folder.name)
print(f'Drive cleanup complete: {len(old_cleanup_targets)} old folders removed.')